In [0]:
-- =============================================================================
-- ALERT QUERY
-- =============================================================================
--
-- ALT-1  Silver gap detection
-- ─────────────────────────────────────────────────────────────────────────────
-- HOW TO SET UP THE ALERT
--   1. Save this as a standalone query named "ALT-1 Silver gap detection"
--   2. Alerts → New alert → select this query → column: silver_events_15min
--   3. Condition: value IS EQUAL TO 0
--   4. Refresh schedule: every 5 minutes
--   5. Notification: email / Slack webhook
--
-- Fires when no Silver events have landed in 15 minutes — almost always
-- means the DLT pipeline has stalled or the Fargate producer has stopped.
-- ─────────────────────────────────────────────────────────────────────────────
SELECT COUNT(*) AS silver_events_15min
FROM wiki_poc.poc.silver_recentchange_enwiki
WHERE ingest_timestamp >= NOW() - INTERVAL 15 MINUTES;


-- =============================================================================
-- SUPPLEMENTARY: Anomaly preview (not a dashboard widget — run ad hoc)
-- =============================================================================
--
-- Shows pages currently spiking above 3σ of their 30-day baseline.
-- Requires at least one run of gold_baseline_job.py to have completed.
-- ─────────────────────────────────────────────────────────────────────────────
SELECT
  g.title,
  g.namespace,
  g.window_start,
  g.edit_count,
  ROUND(b.mean_edit_count, 1)    AS baseline_mean,
  ROUND(b.stddev_edit_count, 1)  AS baseline_stddev,
  ROUND(
    (g.edit_count - b.mean_edit_count) / NULLIF(b.stddev_edit_count, 0),
  2)                             AS z_score
FROM wiki_poc.poc.gold_page_edits_5min g
JOIN wiki_poc.poc.gold_page_edits_baseline b
  ON  g.title     = b.title
  AND g.namespace = b.namespace
  AND b.baseline_date = CURRENT_DATE() - 1
WHERE g.window_start >= NOW() - INTERVAL 1 HOUR
  AND (g.edit_count - b.mean_edit_count) / NULLIF(b.stddev_edit_count, 0) > 3
ORDER BY z_score DESC
LIMIT 20;
